<div align="center">

# Fine-Tuning Qwen 3.5-9B

🌐 **Language:** 🇬🇧 **English**

🤗 **Source:** [Jackrong](https://huggingface.co/Jackrong)

<br>

[![Unsloth](https://img.shields.io/badge/Powered%20by-Unsloth-8A2BE2?style=flat-square)](https://github.com/unslothai/unsloth)
[![Google Colab](https://img.shields.io/badge/Environment-Google%20Colab-F9AB00?style=flat-square&logo=googlecolab&logoColor=white)](https://colab.research.google.com/)
[![PyTorch](https://img.shields.io/badge/Framework-PyTorch-EE4C2C?style=flat-square&logo=pytorch&logoColor=white)](https://pytorch.org/)
[![Hugging Face](https://img.shields.io/badge/Model%20Hub-Hugging%20Face-FFD21E?style=flat-square&logo=huggingface&logoColor=black)](https://huggingface.co/)
[![QLoRA](https://img.shields.io/badge/Technique-QLoRA%20(NF4)-007EC6?style=flat-square)](#)
[![Tool-Calling](https://img.shields.io/badge/Domain-Tool%20Calling-orange?style=flat-square)](#)

</div>

---

## Model & Dataset

- **Base Model**: Qwen3.5-9B (Unsloth 4-bit quantized)
- **Fine-tuning Method**: QLoRA (NF4 + Double Quantization)
- **Dataset**: [younissk/tool-calling-mix](https://huggingface.co/datasets/younissk/tool-calling-mix)
- **Samples**: ~68K (train + validation splits)
- **Tracking**: Weights & Biases


## Before You Start: Required API Keys 🔑

If you are new to Kaggle notebooks, please prepare **two API keys** before running the cells below, and store them in **Kaggle Secrets** first.

### 1. `WANDB_API_KEY`

- This key is used to log in to **Weights & Biases (W&B)**.
- In this notebook, it is used for experiment tracking, logging, and training visualization.
- Without it, the W&B login cell at the beginning will fail.

### 2. `HF_TOKEN`

- This key is used to log in to **Hugging Face**.
- In this notebook, it is mainly needed later if you want to **upload the trained model or GGUF files to Hugging Face Hub**.
- If you only want to train inside Kaggle and do not plan to upload artifacts, this key may not be needed immediately, but it is still recommended to prepare it in advance.

### How to store them in Kaggle Secrets

1. Open your Kaggle notebook.
2. Open the **Secrets** panel on the right side.
3. Add the following secret names exactly as written:
   - `WANDB_API_KEY`
   - `HF_TOKEN`
4. Paste the corresponding value for each key.
5. Save the secrets, then come back and run the notebook.

### Beginner Tip ✨

- Keep the secret **names** exactly the same as the code expects.
- Do not paste API keys directly into notebook code cells.
- Using Kaggle Secrets is the safer and cleaner way to manage credentials.


In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

try:
    wandb_key = user_secrets.get_secret("WANDB_API_KEY")
except:
    print("WANDB_API_KEY not found. W&B logging will be disabled.")
    wandb_key = None

if wandb_key:
    import wandb
    wandb.login(key=wandb_key)
    print("Logged in to W&B successfully!")
else:
    print("W&B login skipped - set WANDB_API_KEY in Kaggle Secrets to enable")

import os
output_directory = "/kaggle/working/"
os.makedirs(output_directory, exist_ok=True)
print(f"Checkpoints will be saved to: {output_directory}")

In [ ]:
%%capture
import os, re, sys

def run_pip_install(cmd):
    """Run pip install with error capture"""
    result = os.popen(cmd + " 2>&1").read()
    if result and ("error" in result.lower() or "exception" in result.lower() or "failed" in result.lower()):
        print(f"pip warning: {result[:500]}", file=sys.stderr)
    return result

if "COLAB_" not in "".join(os.environ.keys()):
    run_pip_install("!pip install unsloth")  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    run_pip_install(f"!pip install sentencepiece protobuf \"datasets==4.3.0\" \"huggingface_hub>=0.34.0\" hf_transfer")
    run_pip_install(f"!pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth")
run_pip_install("!pip install transformers==5.3.0")
run_pip_install("!pip install --no-deps trl==0.22.2")

In [ ]:
import unsloth
from unsloth import FastLanguageModel
import torch
from transformers import BitsAndBytesConfig


# Add special tool-calling tokens to tokenizer vocab
# These tokens enable structured tool call output during training
special_tokens = {
    "additional_special_tokens": [
        "<tool>", "</tool>",
        "<tool_name>", "</tool_name>",
        "<tool_args>", "</tool_args>",
        "<tool_result>", "</tool_result>"
    ]
}

# QLoRA Configuration - NF4 with double quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type = "nf4",
    bnb_4bit_compute_dtype = torch.bfloat16,
    bnb_4bit_use_double_quant = True,
)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Unsloth/Qwen3.5-9B",
    max_seq_length = 16384,
    load_in_4bit = True,
    load_in_8bit = False,
    full_finetuning = False,
    quantization_config = bnb_config,
)

tokenizer.add_special_tokens(special_tokens)
model.resize_token_embeddings(len(tokenizer))

print("Model loaded with QLoRA configuration (NF4 + double quantization)")
print(f"Tokenizer vocab size: {len(tokenizer)}")

In [ ]:
# ============================================================
# LoRA CONFIGURATION CONSTANTS
# ============================================================
# RANDOM_SEED=3407: Chosen for reproducibility. This specific seed
# provides consistent data shuffling and weight initialization.
# 3407 is a prime number, avoiding systematic patterns in splits.
#
# learning_rate=2e-4: Standard learning rate for LoRA fine-tuning.
# 2e-4 balances fast convergence with stability for 4-bit models.
# Lower rates risk slow convergence; higher rates risk divergence.
#
# LoRA rank (r)=64: Dimension of low-rank adaptation matrices.
# 64 provides good capacity for tool-calling task learning while
# keeping parameter overhead manageable (~33M params for 9B model).
# Higher r = more expressiveness but more compute/memory.
RANDOM_SEED = 3407
LEARNING_RATE = 2e-4
LORA_RANK = 64

model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_RANK,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = LORA_RANK,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = RANDOM_SEED,
    use_rslora = True,
    loftq_config = None,
)

print("LoRA adapter configured with QLoRA optimization (RSLoRA enabled)")

## Dataset: younissk/tool-calling-mix 📚

This dataset is designed for fine-tuning language models on tool use / function calling.

### Dataset Statistics

| Split | Samples |
|-------|--------|
| train | 60,600 |
| validation | 7,580 |
| test | 7,580 |
| **Total** | **75,760** |

### Data Composition

- **ToolBench Normalized**: 26.4%
- **xLAM60k**: 26.4%
- **OpenFunctions v1**: 15.2%
- **No-call (Dolly)**: 10.6%
- **No-call (WikiText)**: 10.6%
- **Synthetic Parallel**: 6.6%
- **Others**: 4.2%

### Key Features

- Average 1.34 tool calls per example
- Maximum 24 tool calls in a single example
- Includes non-tool calling examples to prevent catastrophic forgetting
- Tool definitions in JSON schema format


In [ ]:
from datasets import load_dataset

print("Loading tool-calling-mix dataset (all splits)...")

dataset = load_dataset("younissk/tool-calling-mix")

print(f"\nDataset splits: {list(dataset.keys())}")
for split_name, split_data in dataset.items():
    print(f"  {split_name}: {len(split_data)} samples")

print(f"\nColumns: {dataset['train'].column_names}")
print(f"\nSample (first example):")
sample = dataset['train'][0]
print(f"  n_calls: {sample['n_calls']}")
print(f"  difficulty: {sample['difficulty']}")
print(f"  meta_source: {sample['meta_source']}")
print(f"  valid: {sample['valid']}")

In [ ]:
import json
from datasets import concatenate_datasets

MAX_CONTEXT_LENGTH = 16384

def convert_to_training_format(example):
    """Convert tool-calling-mix format to training-ready conversations"""
    try:
        messages = json.loads(example['messages_json'])
        tools = json.loads(example['tools_json'])
        
        conversations = []
        
        # Build system message with tools
        tool_desc = json.dumps(tools, indent=2)
        system_msg = f"""You are a helpful AI assistant. You can use tools to help complete tasks.
Available tools:
{tool_desc}"""
        conversations.append({"role": "system", "content": system_msg})
        
        # Process messages
        for msg in messages:
            role = msg.get('role', '')
            content = msg.get('content', '') or ''
            
            if role == 'system':
                continue  # Skip duplicate system
            elif role == 'user':
                conversations.append({"role": "user", "content": content})
            elif role == 'assistant':
                # Include tool calls in response using spec format
                if msg.get('tool_calls'):
                    for tc in msg['tool_calls']:
                        tool_name = tc.get('function', {}).get('name', 'unknown')
                        tool_args = tc.get('function', {}).get('arguments', '{}')
                        content += f"\n<tool>{tool_name}</tool>\n<tool_name>\n{tool_name}\n</tool_name>\n<tool_args>\n{tool_args}\n</tool_args>\n"
                conversations.append({"role": "assistant", "content": content})
            elif role == 'tool':
                tool_result = msg.get('tool_result', '')
                result_content = f"\n<tool_result>\n{tool_result}\n</tool_result>\n"
                if conversations and conversations[-1]['role'] == 'assistant':
                    conversations[-1]['content'] += result_content
                else:
                    conversations.append({"role": "tool", "content": result_content})
        
        # Validate conversation structure
        if len(conversations) < 2:
            return {"conversations": None}
        if conversations[-1]["role"] != "assistant":
            return {"conversations": None}
        
        return {"conversations": conversations}
        
    except Exception as e:
        return {"conversations": None}

print("Combining train and validation splits...")
combined_dataset = concatenate_datasets([
    dataset['train'],
    dataset['validation']
])
print(f"Combined dataset: {len(combined_dataset)} samples")

print("\nConverting to training format...")
processed = combined_dataset.map(
    convert_to_training_format,
    remove_columns=combined_dataset.column_names,
    keep_in_memory=True,
)

processed = processed.filter(lambda x: x['conversations'] is not None)
print(f"Valid samples after processing: {len(processed)}")

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen3",
)

def apply_chat_template(example):
    """Apply Qwen chat template to create training text"""
    try:
        text = tokenizer.apply_chat_template(
            example['conversations'],
            tokenize=False,
            add_generation_prompt=False,
        )
        return {"text": text}
    except Exception as e:
        return {"text": ""}

print("Applying Qwen chat template...")
dataset_formatted = processed.map(
    apply_chat_template,
    remove_columns=["conversations"],
    keep_in_memory=True,
)

dataset_formatted = dataset_formatted.filter(lambda x: len(x['text']) > 0)
print(f"Samples after formatting: {len(dataset_formatted)}")

print("\nSample output (first 800 chars):")
print(dataset_formatted[0]['text'][:800])

In [ ]:
def check_token_length(example):
    """Filter samples exceeding context window"""
    tokens = tokenizer(
        example['text'],
        truncation=True,
        max_length=MAX_CONTEXT_LENGTH + 1,
        add_special_tokens=False
    )
    return len(tokens['input_ids']) <= MAX_CONTEXT_LENGTH

print(f"Filtering samples longer than {MAX_CONTEXT_LENGTH} tokens...")
dataset_filtered = dataset_formatted.filter(
    check_token_length,
    batched=True,
    batch_size=100,
)

before_count = len(dataset_formatted)
after_count = len(dataset_filtered)
if before_count > 0:
    pct_removed = (before_count - after_count) / before_count * 100
    print(f"Removed {before_count - after_count} samples ({pct_removed:.1f}%)")
else:
    print("Warning: No samples in dataset before filtering")
print(f"Final dataset size: {after_count} samples")

dataset_final = dataset_filtered.shuffle(seed=RANDOM_SEED)
print("Dataset shuffled and ready for training")

In [ ]:
# Validate dataset is non-empty before training
if len(dataset_final) == 0:
    raise ValueError("Dataset is empty after filtering. Check data processing steps.")

from trl import SFTTrainer, SFTConfig

wandb_project_name = "qwen-tool-calling-qlora"

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset_final,
    eval_dataset = None,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 8,
        warmup_ratio = 0.04,
        num_train_epochs = 2,
        learning_rate = LEARNING_RATE,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = RANDOM_SEED,
        save_steps = 100,
        save_total_limit = 2,
        save_strategy = "steps",
        report_to = "wandb" if wandb_key else "none",
        wandb_project = wandb_project_name,
        output_dir = output_directory,
        max_seq_length = MAX_CONTEXT_LENGTH,
        logging_dir = f"{output_directory}/logs",
    ),
)

print(f"Trainer configured:")
print(f"  - Batch size: 4 (per device)")
print(f"  - Gradient accumulation: 8")
print(f"  - Effective batch: 32")
print(f"  - Learning rate: {LEARNING_RATE}")
print(f"  - Epochs: 2")
print(f"  - W&B project: {wandb_project_name}")

In [ ]:
from unsloth.chat_templates import train_on_responses_only

# Use tokenizer chat_template markers instead of hardcoded strings
INSTRUCTION_MARKER = tokenizer.chat_template.split("user\n")[0] + "user\n" if tokenizer.chat_template else "<|im_start|>user\n"
RESPONSE_MARKER = tokenizer.chat_template.split("assistant\n")[0] + "assistant\n" if tokenizer.chat_template else "<|im_start|>assistant\n"

trainer = train_on_responses_only(
    trainer,
    instruction_part = INSTRUCTION_MARKER,
    response_part = RESPONSE_MARKER,
)

print("Training configured to learn only from assistant responses")

In [ ]:
# ============================================================
# DEBUG: Verify label masking
# Set VERIFY=True to enable verification output
# ============================================================
VERIFY = False

if VERIFY:
    print("Verifying label masking...")
    print("\nInput tokens (first 200 chars decoded):")
    print(tokenizer.decode(trainer.train_dataset[0]["input_ids"][:200]))
    print("\nLabels (first 200 chars decoded):")
    label_tokens = [tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[0]["labels"]]
    print(tokenizer.decode(label_tokens[:200]))
else:
    print("Label masking verification skipped (set VERIFY=True to enable)")

In [ ]:
print("Starting training...")
print("=" * 50)

trainer.train()

print("=" * 50)
print("Training completed!")

In [ ]:
save_path = f"{output_directory}qwen_tool_calling_qlora"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"Model saved to: {save_path}")
print("\nFiles saved:")
import os
for f in os.listdir(save_path):
    print(f"  - {f}")

# ============================================================
# Merge LoRA adapters with base model
# ============================================================
print("\nMerging LoRA adapters with base model...")
merged_model = model.merge_and_unload()
print("Merge complete!")

merged_save_path = f"{output_directory}qwen_tool_calling_merged"
merged_model.save_pretrained(merged_save_path)
tokenizer.save_pretrained(merged_save_path)
print(f"Merged model saved to: {merged_save_path}")

## GGUF Export (llama.cpp)

Convert the merged model to GGUF format for local inference using llama.cpp


In [ ]:
# Export merged model to GGUF format using Unsloth
# Unsloth provides a convenient method to export to GGUF
print("Exporting merged model to GGUF format...")
print("This may take several minutes...")

# Method: Use Unsloth's built-in GGUF export
# This creates a Q8_0 quantized GGUF file
try:
    merged_model.save_pretrained_gguf(
        "qwen_tool_calling_q8_0",
        tokenizer,
        quantization_method="q8_0",
    )
    print("GGUF export completed successfully!")
except Exception as e:
    print(f"Unsloth export failed: {e}")
    print("Falling back to llama.cpp convert script method...")
    
    # Fallback: Use llama.cpp convert script
    import subprocess
    import os
    
    # Save merged model in HF format first if not already saved
    if not os.path.exists(merged_save_path):
        merged_model.save_pretrained(merged_save_path)
        tokenizer.save_pretrained(merged_save_path)
    
    # Use llama.cpp convert script
    gguf_path = "qwen_tool_calling_q8_0.gguf"
    result = subprocess.run([
        "python", "-m", "gguf", "scripts.convert_hf_to_gguf",
        merged_save_path,
        "--outfile", gguf_path,
        "--quantize", "q8_0"
    ], capture_output=True, text=True)
    
    if result.returncode == 0:
        print(f"GGUF saved to: {gguf_path}")
    else:
        print(f"llama.cpp conversion failed: {result.stderr}")


In [ ]:
# Optional: Push to Hugging Face Hub
# Uncomment and run if you want to upload the model

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    print("HF_TOKEN found in secrets")
    
    from huggingface_hub import whoami
    user_info = whoami(token=hf_token)
    username = user_info['name']
    repo_id = f"{username}/qwen-tool-calling-qlora"
    
    print(f"\nUploading to: https://huggingface.co/{repo_id}")
    
    merged_model.push_to_hub_merged(
        repo_id,
        tokenizer,
        save_method = "merged_16bit",
        token = hf_token
    )
    
    print(f"Upload complete!")
    
except Exception as e:
    print(f"HF upload skipped: {e}")
    print("Add HF_TOKEN to Kaggle Secrets to enable upload")

## Summary 📊

### Training Complete

| Component | Value |
|-----------|-------|
| Model | Qwen3.5-9B (Unsloth 4-bit) |
| Method | QLoRA (NF4 + Double Quant) |
| LoRA Rank | 64 |
| Dataset | younissk/tool-calling-mix |
| Training Samples | ~67K |
| Epochs | 2 |
| Effective Batch | 32 |

### Next Steps

1. **Test the model** - Use the saved model for inference
2. **Evaluate** - Run on tool-calling benchmarks (BFCL, API-Bank)
3. **Merge and export** - Convert to GGUF for local inference
4. **Iterate** - Adjust hyperparameters based on evaluation
